## Análises do set LEGO Legends of Chima

A escolha do tema LEGO Legends of Chima como foco desta análise foi motivada por características identificadas durante a exploração inicial da base de dados do Rebrickable.

Entre os resultados observados, o conjunto The Ultimate Battle for Chima se destaca pelo elevado número de peças, ficando entre os maiores conjuntos encontrados na base, atrás apenas de World Map e Eiffel Tower. Além disso, o conjunto também aparece entre os que possuem o maior número de minifiguras nos registros analisados.

Essas características tornam o tema um caso interessante para investigar a composição dos conjuntos, a presença e a recorrência de personagens e as relações entre minifiguras e sets ao longo dos anos.

1) Primeiro, vamos conferir como o tema está registrado na tabela "themes".

In [ ]:
query_temas_chima = """
SELECT
    id,
    name,
    parent_id
FROM themes
WHERE name LIKE '%Chima%'
ORDER BY name
"""

spark.sql(query_temas_chima).show(50, truncate=False)

A consulta identificou o tema **Legends of Chima**, registrado na tabela `themes` com o identificador `571`. O campo `parent_id` está nulo, indicando que não há um tema pai informado para esse registro.

Com esse identificador, podemos filtrar os conjuntos associados diretamente ao tema por meio da coluna `theme_id` da tabela `sets`, permitindo concentrar as próximas análises nos conjuntos classificados como Legends of Chima.

2) Vamos agora buscar as figures com personagens mais recorrentes.

In [ ]:
query_personagens_chima = """
WITH minifigs_chima AS (
    SELECT DISTINCT
        s.set_num,
        m.fig_num,
        m.name AS minifig_name
    FROM sets s
    JOIN inventories i
        ON s.set_num = i.set_num
    JOIN inventory_minifigs im
        ON i.id = im.inventory_id
    JOIN minifigs m
        ON im.fig_num = m.fig_num
    WHERE s.theme_id = 571
),

personagens AS (
    SELECT
        set_num,
        fig_num,
        CASE
            WHEN name_base LIKE '%Laval%' THEN 'Laval'
            WHEN name_base LIKE '%Cragger%' THEN 'Cragger'
            ELSE name_base
        END AS personagem
    FROM (
        SELECT
            set_num,
            fig_num,
            TRIM(
                REGEXP_REPLACE(
                    minifig_name,
                    '( with .*| in .*|,.*| \\(.*\\))$',
                    ''
                )
            ) AS name_base
        FROM minifigs_chima
    )
)

SELECT
    personagem,
    COUNT(DISTINCT set_num) AS total_sets
FROM personagens
GROUP BY personagem
ORDER BY total_sets DESC, personagem ASC
LIMIT 30
"""

resultado_personagens_chima = spark.sql(query_personagens_chima)
resultado_personagens_chima.show(30, truncate=False)

Os resultados indicam que os três primeiros colocados no ranking aparecem em mais conjuntos do que Laval, personagem protagonista da série temática e escolhido como foco das análises seguintes. Esse resultado é interessante porque mostra que a recorrência de uma minifigura nos produtos da franquia não necessariamente coincide com a importância narrativa atribuída a um personagem.

É importante considerar, entretanto, que o ranking depende das regras utilizadas para agrupar as variações de nomes e representa a presença dos personagens nos conjuntos registrados na base, não sua popularidade ou importância oficial na história.

3) Vamos agora encontrar as figures do personagem protagonista, Laval:

In [ ]:
query_laval = """
SELECT
    fig_num,
    name AS minifig_name
FROM minifigs
WHERE name LIKE '%Laval%'
ORDER BY name
"""

resultado_laval = spark.sql(query_laval)

resultado_laval.show(truncate=False)

total_versoes_laval = resultado_laval.count()

print(f"Total de versões do Laval: {total_versoes_laval}")

A consulta identificou oito registros de minifiguras relacionados ao personagem Laval. Os nomes apresentam variações associadas a roupas, armaduras e outros elementos visuais, permitindo observar como o personagem foi representado em diferentes configurações nos produtos da franquia.

4) Analisar quais sets contém as figures do Laval encontradas anteriormente:

In [ ]:
query_sets_com_laval = """
SELECT DISTINCT
    s.set_num,
    s.name AS set_name,
    s.year,
    m.name AS minifig_name
FROM sets s
JOIN inventories i
    ON s.set_num = i.set_num
JOIN inventory_minifigs im
    ON i.id = im.inventory_id
JOIN minifigs m
    ON im.fig_num = m.fig_num
WHERE m.name LIKE '%Laval%'
ORDER BY year, set_name
"""

resultado_sets_com_laval = spark.sql(query_sets_com_laval)

resultado_sets_com_laval.show(50, truncate=False)

query_contagem_sets_laval = """
SELECT
    COUNT(DISTINCT s.set_num) AS total_sets_com_laval
FROM sets s
JOIN inventories i
    ON s.set_num = i.set_num
JOIN inventory_minifigs im
    ON i.id = im.inventory_id
JOIN minifigs m
    ON im.fig_num = m.fig_num
WHERE m.name LIKE '%Laval%'
"""

resultado_contagem_sets_laval = spark.sql(query_contagem_sets_laval)

resultado_contagem_sets_laval.show()

A análise considera os conjuntos distintos associados às minifiguras encontradas nos registros de inventário disponíveis. Dessa forma, diferentes versões de Laval presentes no mesmo conjunto não aumentam a contagem de conjuntos.

Observou-se também que um dos conjuntos contém duas minifiguras diferentes relacionadas ao personagem, demonstrando que a presença de um personagem em um set não implica necessariamente a inclusão de apenas uma de suas versões.

5) Vamos agora analisar qual figure acompanha Laval mais frequentemente:

In [ ]:
query_figures_com_laval = """
WITH sets_com_laval AS (
    SELECT DISTINCT
        s.set_num
    FROM sets s
    JOIN inventories i
        ON s.set_num = i.set_num
    JOIN inventory_minifigs im
        ON i.id = im.inventory_id
    JOIN minifigs m
        ON im.fig_num = m.fig_num
    WHERE m.name LIKE '%Laval%'
)

SELECT
    m.fig_num,
    m.name AS minifig_name,
    COUNT(DISTINCT s.set_num) AS total_sets
FROM sets s
JOIN inventories i
    ON s.set_num = i.set_num
JOIN inventory_minifigs im
    ON i.id = im.inventory_id
JOIN minifigs m
    ON im.fig_num = m.fig_num
JOIN sets_com_laval sl
    ON s.set_num = sl.set_num
WHERE m.name NOT LIKE '%Laval%'
GROUP BY
    m.fig_num,
    m.name
ORDER BY
    total_sets DESC,
    minifig_name
LIMIT 10
"""

resultado_figures_com_laval = spark.sql(query_figures_com_laval)

# Converter para Pandas para utilizar no gráfico
top_figures_laval = resultado_figures_com_laval.toPandas()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 9))

plt.pie(
    top_figures_laval["total_sets"],
    labels=top_figures_laval["minifig_name"],
    autopct="%1.1f%%",
    startangle=90
)

plt.title("10 minifiguras mais recorrentes nos sets com Laval")
plt.tight_layout()
plt.show()

Analisando os registros encontrados, destacam-se variantes associadas a Cragger, personagem que possui papel de antagonista na franquia.

Entretanto, como cada versão de uma minifigura é contabilizada separadamente, a frequência total de um personagem pode estar distribuída entre diferentes registros. Por esse motivo, a análise seguinte buscará agrupar as variações por personagem, permitindo uma comparação mais consistente de sua recorrência nos conjuntos com Laval.

6) Sets que contém Laval e Cragger juntos:

In [ ]:
query_sets_laval_cragger = """
WITH personagens_por_set AS (
    SELECT DISTINCT
        s.set_num,
        s.name AS set_name,
        s.year,
        m.name AS minifig_name
    FROM sets s
    JOIN inventories i
        ON s.set_num = i.set_num
    JOIN inventory_minifigs im
        ON i.id = im.inventory_id
    JOIN minifigs m
        ON im.fig_num = m.fig_num
    WHERE m.name LIKE '%Laval%'
       OR m.name LIKE '%Cragger%'
)

SELECT
    set_num,
    set_name,
    year
FROM personagens_por_set
GROUP BY
    set_num,
    set_name,
    year
HAVING
    MAX(CASE WHEN minifig_name LIKE '%Laval%' THEN 1 ELSE 0 END) = 1
    AND
    MAX(CASE WHEN minifig_name LIKE '%Cragger%' THEN 1 ELSE 0 END) = 1
ORDER BY year, set_name
"""

resultado_sets_laval_cragger = spark.sql(query_sets_laval_cragger)

resultado_sets_laval_cragger.show(50, truncate=False)

In [ ]:
import matplotlib.pyplot as plt

# Converter o resultado do Spark para Pandas
df_laval_cragger = resultado_sets_laval_cragger.toPandas()

# Contar os sets únicos por ano
sets_por_ano = (
    df_laval_cragger.groupby("year")["set_num"]
    .nunique()
    .sort_index()
)

# Criar o gráfico
plt.figure(figsize=(8, 5))
plt.bar(sets_por_ano.index.astype(str), sets_por_ano.values)

plt.title("Sets com Laval e Cragger por ano")
plt.xlabel("Ano de lançamento")
plt.ylabel("Quantidade de sets")
plt.tight_layout()
plt.show()

A análise identificou 6 conjuntos distintos que contêm minifiguras relacionadas a Laval e Cragger nos registros de inventário disponíveis.

Esses conjuntos estão distribuídos igualmente entre os anos de 2013 a 2015, com dois conjuntos identificados em cada ano.